# 01 Data Check

Validate the local `underlying_1m` DuckDB view after running Massive 1-minute aggregate ingestion.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from odte_flow_lab.config import get_settings
from odte_flow_lab.data.duckdb_catalog import connect, register_underlying_view

settings = get_settings()
con = connect()
register_underlying_view(con)
settings.duckdb_path

## Row Counts And Session Coverage

In [ ]:
con.sql("""
SELECT
    symbol,
    trade_date,
    COUNT(*) AS rows,
    MIN(bar_start_et) AS first_bar_et,
    MAX(bar_start_et) AS last_bar_et,
    SUM(CASE WHEN SUBSTR(bar_start_et, 12, 8) BETWEEN '09:30:00' AND '15:59:00' THEN 1 ELSE 0 END) AS regular_session_rows
FROM underlying_1m
GROUP BY symbol, trade_date
ORDER BY trade_date DESC, symbol
""").df()

## Duplicate Bars

In [ ]:
con.sql("""
SELECT symbol, bar_start_utc, COUNT(*) AS duplicate_count
FROM underlying_1m
GROUP BY symbol, bar_start_utc
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC, symbol, bar_start_utc
""").df()

## Missing Regular-Session Minute Check

A full regular session usually has 390 one-minute bars. Holidays and half-days should be reviewed manually.

In [ ]:
con.sql("""
WITH counts AS (
    SELECT
        symbol,
        trade_date,
        SUM(CASE WHEN SUBSTR(bar_start_et, 12, 8) BETWEEN '09:30:00' AND '15:59:00' THEN 1 ELSE 0 END) AS regular_session_rows
    FROM underlying_1m
    GROUP BY symbol, trade_date
)
SELECT *, 390 - regular_session_rows AS missing_vs_full_session
FROM counts
WHERE regular_session_rows <> 390
ORDER BY trade_date DESC, symbol
""").df()

## OHLC, Volume, And VWAP Sanity

In [ ]:
con.sql("""
SELECT
    symbol,
    trade_date,
    SUM(CASE WHEN low > open OR low > close OR high < open OR high < close OR low > high THEN 1 ELSE 0 END) AS bad_ohlc_rows,
    SUM(CASE WHEN volume IS NULL OR volume < 0 THEN 1 ELSE 0 END) AS bad_volume_rows,
    SUM(CASE WHEN vwap IS NULL THEN 1 ELSE 0 END) AS null_massive_vwap_rows
FROM underlying_1m
GROUP BY symbol, trade_date
ORDER BY trade_date DESC, symbol
""").df()